# Train the Stage 2 semantic change model (SECOND-CC)

**Goal:** given a before/after optical pair, predict **where** the scene changed and the **land-cover class of every changed pixel at each date** -- the model `models/change_detection/semantic_change_tool.py` wraps for inference. It replaces Stage 1's training-free pixel differencing, which can say *that* something changed but not *which* land-cover class gained or lost area, so it can't answer the spec's own representative query "Has the built-up area increased, decreased, or remained unchanged?".

**Data:** [SECOND-CC](https://zenodo.org/records/16937571) (Zenodo 16937571, CC-BY-4.0, one 2.5GB zip). Downloaded straight from Zenodo inside this notebook (about 12 minutes from Kaggle's network; a home connection was throttled to ~100KB/s), unless you've attached it as an input dataset -- the cell below prefers a mounted copy.

## What the real data looks like (checked directly before any of this was written)

* **256x256 RGB pairs.** `{train,val,test}/rgb/{A,B}/<name>.png` (A = before, B = after), matching RGB-colour-coded semantic maps in `sem/{A,B}`, plus one caption JSON. 10,855 entries = 6,041 originals (train 4,219 / val 595 / test 1,227) + 4,814 offline-augmented copies (`_random_augment`, exactly one per train and val original; none in test).
* **The offline-augmented copies are NOT trained on.** Found the hard way: the first Kaggle run died on this notebook's own palette check -- 1,812,124 of 39,321,600 pixels (4.6%) in a 300-pair random sample of the train split were not one of the seven colours, although a local check of 163 pairs had found exactly seven (those 163 were all originals). Ten augmented pairs (seven copies of changed scenes, three of no-change scenes whose all-white maps are unaffected) fetched straight from the zip and compared with their originals show why: each copy is a random 90-degree rotation/flip of the original, applied consistently to the images and the label maps (the transformed label equals the same rotation/flip of the original's), and in most of the copies of changed scenes (5 of the 7 inspected) an additive **+80 brightness shift that was applied to the label maps as well as the imagery** -- palette colours come out as `clip(colour + 80)`, e.g. buildings `(128,0,0)` -> `(208,80,80)`. That is recoverable, but the copies add no scene content beyond what the online rotation/flip and photometric jitter below already provide, and decoding a bugged shift correctly for all 4,219 of them without being able to inspect them all isn't worth the risk. So training uses originals only, and the label audit below checks every remaining map instead of a sample.
* **The semantic maps only label the CHANGED regions.** White `(255,255,255)` means "no change" and is identical in the A and B maps (0 of 10.7M pixels differed across 163 sampled pairs); every other pixel carries one of six classes, and those seven colours are the only ones in the maps. So a plain per-image land-cover segmenter can't be trained from these labels -- roughly 85% of every image is unlabeled. What the labels *do* fully determine is the **net area change per class**: unchanged pixels have the same class at both dates and cancel out of (area at t2) - (area at t1). That is why this is trained as semantic *change* detection: a change mask plus a class map for each date, with the class loss applied only where the labels exist.
* **Which green is which** was decided statistically, not by eye (an eyeball guess from three samples got it backwards): among pairs containing only the bright green `(0,255,0)`, 89% of captions mention "tree(s)" vs 33% for pairs containing only the dark green `(0,128,0)`, whose captions say sparse vegetation / green field / farmland. The imagery agrees: pixels labeled `(0,255,0)` are darker (mean luminance 75.7 vs 85.2), rougher (5x5 local std 11.1 vs 8.9) and greener than `(0,128,0)` ones -- tree canopy vs. smooth grass/fields. So `(0,255,0)` = trees and `(0,128,0)` = low vegetation.
* **Class balance** (share of labeled pixels over the 3,860 train originals, as measured by the audit cell below -- an earlier 163-pair sample had under-counted the two rare classes): bare ground 33%, buildings 31%, low vegetation 24%, trees 10%, water 1.3%, playground 0.8%. Water and playground are rare and will be the weakest classes; buildings -- the class the headline query is about -- is the best supported.
* **Split leakage, found and handled.** 246 crops (scene id + crop index) appear in both train and test: the *same physical pair*, once time-reversed (`_ters_`), so a model trained on one has seen the other's pixels. Any train/val entry whose crop also appears in a later split is dropped below; the official test set stays intact, so numbers remain comparable. Different crops of one larger scene can still sit in different splits -- that's the dataset authors' own protocol and is left alone.

## Model

`SiameseSCDNet`: one ImageNet-pretrained ResNet34 encoder shared by both dates; **one semantic decoder shared by both dates** (the same land-cover appearance model at t1 and t2); a separate change decoder over per-level fused features `[fA, fB, |fA-fB|]`. Outputs a change logit plus K-class logits per date. Loss = BCE + Dice on the change mask (all pixels) + class-weighted cross-entropy on each date's classes, **ignoring unchanged pixels** (they have no label).

## Metrics

* **SeK / Score** -- the standard SECOND semantic-change metrics (Separated Kappa, and 0.3*mIoU + 0.7*SeK).
* **Class-delta metrics** -- what the tool actually reports to a user: per-class net area change (as a share of the scene) predicted vs. true, and whether the *direction* for buildings (increased / decreased / unchanged) is right.

Run cells top to bottom. Kaggle: enable **Internet** and a **GPU**.

## 0. GPU compatibility check

**Found live, via an actual failed run on Kaggle**: this session's preinstalled PyTorch build
(2.10.0+cu128) only supports CUDA compute capabilities sm_70 and up -- it silently drops support
for the Pascal-generation **P100** (sm_60), one of the two GPU types Kaggle itself still offers
here (T4x2 or P100, per the note below). Landing on a P100 crashed training ~40 seconds in with
`CUDA error: no kernel image is available for execution on the device` -- a real run, not a
hypothetical. The cell below detects the actual GPU via `nvidia-smi` (no torch import needed yet,
so this runs before torch's own compute-capability list is fixed for the process) and reinstalls
a CUDA 11.8 build if the assigned GPU isn't in the preinstalled build's supported list -- CUDA 11.8
wheels cover Pascal through Hopper, so this works regardless of which GPU Kaggle happens to assign.

In [ ]:
import subprocess, sys

try:
    cc_raw = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=compute_cap", "--format=csv,noheader"], text=True
    ).strip().splitlines()[0]
    major, minor = cc_raw.split(".")
    needed_sm = f"sm_{major}{minor}"
except Exception as e:
    needed_sm = None
    print(f"Could not query GPU compute capability via nvidia-smi ({e}) -- skipping the compatibility check.")

if needed_sm:
    # Check the INSTALLED build's supported architectures in a SEPARATE PROCESS, not an in-process
    # `import torch` -- Python caches imports in sys.modules, so even an aliased/deleted in-process
    # import here would make a LATER `import torch` in the next cell silently return the stale
    # cached module instead of a fresh one. Confirmed live: an earlier version of this cell did
    # `import torch as _torch_probe`, and the kernel died ~80s after reinstalling -- the old torch
    # stayed resident in this process while its .so files got replaced out from under it on disk.
    check = subprocess.run(
        [sys.executable, "-c",
         "import torch; print(' '.join(torch.cuda.get_arch_list()) if torch.cuda.is_available() else '')"],
        capture_output=True, text=True,
    )
    supported = check.stdout.split()
    if needed_sm not in supported:
        print(f"GPU needs {needed_sm}, not in the preinstalled torch build's supported list "
              f"{supported} -- reinstalling a CUDA 11.8 build (covers Pascal through Hopper)...")
        # Uninstall the whole torch/torchvision/torchaudio trio first, then install all three
        # together from the SAME cu118 index in one resolution -- reinstalling `torch` alone
        # left mismatched torchvision/nccl versions behind, which crashed two live runs with
        # unrelated-looking errors (undefined symbol ncclCommShrink; aten.OpaqueObject not
        # registered) that were actually both this same root cause from a different angle.
        subprocess.run(["pip", "uninstall", "-y", "-q", "torch", "torchvision", "torchaudio"], check=True)
        subprocess.run(
            ["pip", "install", "-q", "torch", "torchvision", "torchaudio",
             "--index-url", "https://download.pytorch.org/whl/cu118"],
            check=True,
        )
        print("Reinstalled. The check above ran in a subprocess, so THIS process has never "
              "imported torch itself -- the next cell's `import torch` will be a genuinely fresh "
              "import, not a cached one.")
    else:
        print(f"GPU compute capability {needed_sm} already supported by the preinstalled build.")

In [ ]:
import torch
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
for i in range(torch.cuda.device_count()):
    print(" -", torch.cuda.get_device_name(i))

## 1. Setup

In [ ]:
!pip install -q segmentation-models-pytorch

In [ ]:
import os, sys, json, re, math, glob, time, random, hashlib, zipfile, collections
import numpy as np
import requests
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# Local smoke-test hooks -- never set on Kaggle. SCD_DATA_ROOT points at a folder that already
# contains SECOND-CC-AUG/; SCD_SMOKE_TEST=1 shrinks epochs/batches so the whole notebook can be run
# end to end on a laptop CPU against a handful of real pairs before spending any GPU time.
SMOKE_TEST = bool(os.environ.get("SCD_SMOKE_TEST"))
DATA_ROOT = os.environ.get("SCD_DATA_ROOT")

SEED = 0
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cuda":
    torch.backends.cudnn.benchmark = True
print("device:", DEVICE, "| smoke test:", SMOKE_TEST)

## 2. Classes and the model

This cell is **duplicated verbatim** from `models/change_detection/semantic_change_tool.py` (its `SHARED MODEL BLOCK` -- the notebook can't import from the repo, and each `models/*/` script stays standalone). `backend/tests/test_semantic_change.py` fails if the two copies drift.

In [ ]:
# --- BEGIN SHARED MODEL BLOCK -----------------------------------------------------------------
# Duplicated verbatim in notebooks/kaggle_finetune_change_segmentation_second.ipynb (each models/*/
# script stays standalone -- see CLAUDE.md -- and a Kaggle notebook can't import from this repo).
# backend/tests/test_semantic_change.py FAILS if the two copies drift, so edit both together.

# Class index -> (name, color in SECOND-CC's RGB-coded semantic maps). White (255,255,255) is NOT a
# class: it marks "no change" and is shared by both dates' maps (0 of 10.7M pixels differed between
# the A and B maps across 163 sampled pairs). The map contains exactly these 7 colors, nothing else.
# The two greens are easy to swap by eye and were, at first: (0,255,0)=trees and (0,128,0)=low
# vegetation is what the captions say -- among pairs containing only the bright green, 89% of
# captions mention "tree(s)" vs 33% for pairs containing only the dark green, whose captions talk
# about sparse vegetation / green fields / farmland instead.
CLASS_NAMES = ["water", "bare ground", "low vegetation", "trees", "buildings", "playground"]
CLASS_COLORS_RGB = [(0, 0, 255), (128, 128, 128), (0, 128, 0), (0, 255, 0), (128, 0, 0), (255, 0, 0)]
NO_CHANGE_COLOR_RGB = (255, 255, 255)
IGNORE_INDEX = 255  # semantic-loss ignore label for unchanged pixels
NUM_CLASSES = len(CLASS_NAMES)


class ConvBNReLU(nn.Sequential):
    def __init__(self, in_ch, out_ch):
        super().__init__(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )


class DecoderBlock(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch):
        super().__init__()
        self.conv1 = ConvBNReLU(in_ch + skip_ch, out_ch)
        self.conv2 = ConvBNReLU(out_ch, out_ch)

    def forward(self, x, skip=None):
        x = F.interpolate(x, scale_factor=2.0, mode="nearest")
        if skip is not None:
            x = torch.cat([x, skip], dim=1)
        return self.conv2(self.conv1(x))


class UNetDecoder(nn.Module):
    """U-Net decoder over a 5-level encoder pyramid: `feats` = [input, f1(H/2), f2(H/4), f3(H/8),
    f4(H/16), f5(H/32)] as returned by smp's get_encoder; the input-resolution entry is unused."""

    def __init__(self, enc_channels, decoder_channels=(256, 128, 64, 32, 16)):
        super().__init__()
        c1, c2, c3, c4, c5 = enc_channels[1:]
        d1, d2, d3, d4, d5 = decoder_channels
        self.b1 = DecoderBlock(c5, c4, d1)
        self.b2 = DecoderBlock(d1, c3, d2)
        self.b3 = DecoderBlock(d2, c2, d3)
        self.b4 = DecoderBlock(d3, c1, d4)
        self.b5 = DecoderBlock(d4, 0, d5)
        self.out_channels = d5

    def forward(self, feats):
        f1, f2, f3, f4, f5 = feats[1:]
        x = self.b1(f5, f4)
        x = self.b2(x, f3)
        x = self.b3(x, f2)
        x = self.b4(x, f1)
        return self.b5(x, None)


class SiameseSCDNet(nn.Module):
    """(before, after) -> (change logit [N,1,H,W], semantic logits before [N,K,H,W], semantic logits
    after [N,K,H,W]). One ResNet encoder shared by both dates; ONE semantic decoder shared by both
    dates (the same land-cover appearance model at t1 and t2); a separate change decoder over
    per-level fused features [fA, fB, |fA-fB|]."""

    def __init__(self, encoder_name="resnet34", encoder_weights=None, num_classes=NUM_CLASSES):
        super().__init__()
        from segmentation_models_pytorch.encoders import get_encoder

        self.encoder = get_encoder(encoder_name, in_channels=3, depth=5, weights=encoder_weights)
        ch = list(self.encoder.out_channels)
        self.sem_decoder = UNetDecoder(ch)
        self.fuse = nn.ModuleList(
            [
                nn.Sequential(nn.Conv2d(3 * c, c, 1, bias=False), nn.BatchNorm2d(c), nn.ReLU(inplace=True))
                for c in ch[1:]
            ]
        )
        self.change_decoder = UNetDecoder(ch)
        self.sem_head = nn.Conv2d(self.sem_decoder.out_channels, num_classes, 3, padding=1)
        self.change_head = nn.Conv2d(self.change_decoder.out_channels, 1, 3, padding=1)

    def forward(self, a, b):
        n = a.shape[0]
        feats = self.encoder(torch.cat([a, b], dim=0))  # one pass for both dates
        sem = self.sem_head(self.sem_decoder(feats))  # [2N,K,H,W], shared weights
        fa = [f[:n] for f in feats]
        fb = [f[n:] for f in feats]
        fused = [fa[0]] + [m(torch.cat([x, y, (x - y).abs()], dim=1)) for m, x, y in zip(self.fuse, fa[1:], fb[1:])]
        change = self.change_head(self.change_decoder(fused))
        return change, sem[:n], sem[n:]


# --- END SHARED MODEL BLOCK -------------------------------------------------------------------

## 3. Get SECOND-CC

Prefers a copy mounted under `/kaggle/input` (attach the dataset if you've uploaded one, to skip the download on re-runs). Otherwise downloads the zip from Zenodo into scratch space (`/kaggle/temp` if that directory exists, else `/tmp` -- it was `/tmp` on the T4 image this ran on -- so the 43k extracted PNGs don't end up in this run's saved output) with **Range-request resume + retry/backoff** (a plain streaming download got dropped mid-file when this was first tried from a laptop), then verifies it against the md5 Zenodo publishes and extracts it.

In [ ]:
ZENODO_RECORD, ZIP_NAME = "16937571", "SECOND-CC-AUG.zip"

def find_mounted():
    hits = glob.glob("/kaggle/input/**/SECOND-CC-AUG.json", recursive=True)
    return os.path.dirname(hits[0]) if hits else None

def download_resumable(url, dest, expected_size, max_attempts=60):
    part = dest + ".part"
    last_report = 0
    for attempt in range(1, max_attempts + 1):
        have = os.path.getsize(part) if os.path.exists(part) else 0
        if have == expected_size:
            break
        headers = {"Range": f"bytes={have}-"} if have else {}
        try:
            with requests.get(url, stream=True, headers=headers, timeout=60) as r:
                if r.status_code == 429:
                    time.sleep(int(r.headers.get("Retry-After", 30)))
                    continue
                r.raise_for_status()
                if have and r.status_code != 206:
                    have = 0  # server ignored the Range header -> restart
                with open(part, "ab" if have else "wb") as f:
                    for chunk in r.iter_content(1 << 20):
                        f.write(chunk)
                        if f.tell() - last_report >= 250 * (1 << 20):
                            last_report = f.tell()
                            print(f"  {f.tell() / 1e9:.2f} / {expected_size / 1e9:.2f} GB", flush=True)
        except requests.RequestException as e:
            print(f"  connection dropped ({type(e).__name__}) -- resuming", flush=True)
        time.sleep(min(30, 2 ** min(attempt, 5)))
    assert os.path.exists(part) and os.path.getsize(part) == expected_size, "download incomplete"
    os.replace(part, dest)

# The zip + its 43k extracted PNGs go in scratch space, NOT /kaggle/working: everything under
# /kaggle/working is saved as the run's output, and a huge output directory makes retrieving the
# one file we care about (the checkpoint) painfully slow -- found the hard way on the grounding v3 run.
SCRATCH_DIR = "/kaggle/temp" if os.path.isdir("/kaggle/temp") else "/tmp"

def fetch_second_cc(work=SCRATCH_DIR):
    meta = requests.get(f"https://zenodo.org/api/records/{ZENODO_RECORD}", timeout=30).json()
    f = next(x for x in meta["files"] if x["key"] == ZIP_NAME)
    zip_path = os.path.join(work, ZIP_NAME)
    t0 = time.time()
    download_resumable(f["links"]["self"], zip_path, f["size"])
    print(f"downloaded in {time.time() - t0:.0f}s; verifying md5 ...")
    md5 = hashlib.md5()
    with open(zip_path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            md5.update(chunk)
    published = f["checksum"].removeprefix("md5:")
    assert md5.hexdigest() == published, f"md5 mismatch: {md5.hexdigest()} != {published}"
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(work)
    os.remove(zip_path)  # free the 2.5GB now that it's extracted
    return os.path.join(work, "SECOND-CC-AUG")

if DATA_ROOT:
    ROOT = os.path.join(DATA_ROOT, "SECOND-CC-AUG")
else:
    mounted = find_mounted()
    ROOT = mounted if mounted else fetch_second_cc()
assert os.path.exists(os.path.join(ROOT, "SECOND-CC-AUG.json")), f"SECOND-CC-AUG.json not found under {ROOT}"
print("ROOT =", ROOT)

## 4. Splits, with the crop-level leak removed

Training uses **original entries only** (see the top of this notebook for why the `_random_augment` copies are excluded). Filenames look like `00013_ters_2_0_random_augment.png`: scene id, optional `png<k>` tag, optional `ters_` (time-reversed), crop index, a counter, optional `_random_augment`. The leak key is **(scene id, tag, crop)** -- deliberately ignoring `ters_` and `_random_augment`, since those are exactly the variants that put the same physical pair on both sides of a split.

In [ ]:
index = json.load(open(os.path.join(ROOT, "SECOND-CC-AUG.json")))["images"]

FN_RX = re.compile(r"^(?P<id>\d+)(?P<tag>png\d+)?_(?P<ters>ters_)?(?P<crop>\d+)_(?P<k>\d+)(?P<aug>_random_augment)?(?:_(?P<extra>\d+))?\.png$")

def crop_key(filename):
    m = FN_RX.match(filename)
    assert m, f"unparseable filename: {filename}"
    return (m["id"], m["tag"] or "", m["crop"])

def is_aug(filename):
    return "_random_augment" in filename

by_split = {s: [x for x in index if x["split"] == s] for s in ("train", "val", "test")}
test_keys = {crop_key(x["filename"]) for x in by_split["test"]}
val_keys = {crop_key(x["filename"]) for x in by_split["val"]}

train_entries = [x for x in by_split["train"] if not is_aug(x["filename"]) and crop_key(x["filename"]) not in test_keys | val_keys]
val_all = [x for x in by_split["val"] if crop_key(x["filename"]) not in test_keys]
val_entries = [x for x in val_all if not is_aug(x["filename"])]  # validate on originals only
test_entries = by_split["test"]  # untouched -- comparable to published numbers

n_train_aug = sum(is_aug(x["filename"]) for x in by_split["train"])
print(f"train: {len(by_split['train'])} -> {len(train_entries)}  (dropped {n_train_aug} offline-augmented copies, and {len(by_split['train']) - n_train_aug - len(train_entries)} originals sharing a crop with val/test)")
print(f"val  : {len(by_split['val'])} -> {len(val_entries)}  (dropped {len(by_split['val']) - len(val_all)} sharing a crop with test; evaluating originals only)")
print(f"test : {len(test_entries)}")

# Prove no crop leaks across splits after filtering.
tr_k = {crop_key(x["filename"]) for x in train_entries}
va_k = {crop_key(x["filename"]) for x in val_entries}
te_k = {crop_key(x["filename"]) for x in test_entries}
assert not (tr_k & te_k) and not (tr_k & va_k) and not (va_k & te_k), "crop-level leak remains"
print("no crop shared between train / val / test")

if SMOKE_TEST:  # only keep entries whose four files exist locally
    def have(x):
        return all(os.path.exists(f"{ROOT}/{x['split']}/{k}/{s}/{x['filename']}") for k in ("rgb", "sem") for s in ("A", "B"))
    train_entries = [x for x in train_entries if have(x)][:24]
    val_entries = [x for x in val_entries if have(x)][:8]
    test_entries = [x for x in test_entries if have(x)][:8]
    print(f"SMOKE: train={len(train_entries)} val={len(val_entries)} test={len(test_entries)}")
    assert train_entries and val_entries and test_entries, "smoke test needs at least one local pair per split"

## 5. Label decoding, sanity-checked against the real maps

`decode_sem` maps the RGB-coded map to class indices, `IGNORE_INDEX` for unchanged pixels -- which also means a pixel in an *unexpected* colour would silently be read as "no change". So the audit below decodes **every** label map in every split (about half a minute -- much cheaper than discovering a problem mid-training, which is what a 300-pair sample did on the first run) and checks what the markdown above claims: only the seven known colours occur, and "white" is the same in A and B. Train/val entries with any unexpected pixel, or a missing file, are dropped (and must stay a small minority); the test split is never filtered, so its numbers stay comparable -- it only has to pass. The same pass measures label frequencies for the loss weights.

In [ ]:
WHITE = np.array(NO_CHANGE_COLOR_RGB, dtype=np.uint8)
COLORS = [np.array(c, dtype=np.uint8) for c in CLASS_COLORS_RGB]
IMG_SIZE = 256

def decode_sem(rgb):
    labels = np.full(rgb.shape[:2], IGNORE_INDEX, dtype=np.uint8)
    for idx, color in enumerate(COLORS):
        labels[(rgb == color).all(axis=-1)] = idx
    return labels

def read_sem(split, side, filename):
    return np.asarray(Image.open(f"{ROOT}/{split}/sem/{side}/{filename}").convert("RGB"))

def audit_labels(entries, name):
    counts = np.zeros(NUM_CLASSES, dtype=np.int64)
    unknown_px = white_mismatch = total_px = blank_pairs = 0
    clean, dirty, missing = [], [], []
    for x in entries:
        paths = [f"{ROOT}/{x['split']}/{k}/{s}/{x['filename']}" for k in ("rgb", "sem") for s in ("A", "B")]
        if not all(os.path.exists(p) for p in paths):
            missing.append(x)
            continue
        a, b = read_sem(x["split"], "A", x["filename"]), read_sem(x["split"], "B", x["filename"])
        la, lb = decode_sem(a), decode_sem(b)
        white_a, white_b = (a == WHITE).all(-1), (b == WHITE).all(-1)
        unk = int(((la == IGNORE_INDEX) & ~white_a).sum() + ((lb == IGNORE_INDEX) & ~white_b).sum())
        unknown_px += unk
        white_mismatch += int((white_a != white_b).sum())
        total_px += white_a.size
        blank_pairs += int(white_a.all() and white_b.all())
        counts += np.bincount(la[la != IGNORE_INDEX], minlength=NUM_CLASSES) + np.bincount(lb[lb != IGNORE_INDEX], minlength=NUM_CLASSES)
        (dirty if unk else clean).append(x)
    print(f"{name:5s}: {len(entries)} entries | unrecognised-colour pixels {unknown_px} of {2 * total_px} "
          f"| white(A)!=white(B) pixels {white_mismatch} of {total_px} | all-white (no-change) pairs {blank_pairs} "
          f"| entries with unexpected colours {len(dirty)}, missing files {len(missing)}")
    return clean, dirty, missing, counts, unknown_px, white_mismatch, total_px

train_clean, train_dirty, train_missing, counts, unk_tr, mism_tr, tot_tr = audit_labels(train_entries, "train")
val_clean, val_dirty, val_missing, _, unk_va, mism_va, tot_va = audit_labels(val_entries, "val")
_, test_dirty, test_missing, _, unk_te, mism_te, tot_te = audit_labels(test_entries, "test")

for nm, kept, bad in (("train", train_clean, train_dirty + train_missing), ("val", val_clean, val_dirty + val_missing)):
    assert len(bad) <= 0.02 * (len(kept) + len(bad)), f"{nm}: {len(bad)} of {len(kept) + len(bad)} entries unusable -- the palette assumption is wrong, check CLASS_COLORS_RGB"
assert not test_missing, f"test entries with missing files would crash the final evaluation: {[x['filename'] for x in test_missing[:5]]}"
assert unk_te < 1e-3 * 2 * tot_te, "test maps contain many colours outside the 7-colour palette -- check CLASS_COLORS_RGB"
for nm, mism, tot in (("train", mism_tr, tot_tr), ("val", mism_va, tot_va), ("test", mism_te, tot_te)):
    assert mism < 1e-3 * tot, f"{nm}: 'no change' isn't shared between A and B -- the change-mask assumption is wrong"
train_entries, val_entries = train_clean, val_clean
print(f"using train={len(train_entries)} val={len(val_entries)} test={len(test_entries)}")

freq = counts / max(counts.sum(), 1)
print("labeled-pixel share per class:", {n: round(float(f), 4) for n, f in zip(CLASS_NAMES, freq)})
# Mild inverse-frequency weights (square-rooted, clipped) so water/playground aren't ignored outright
# without letting a handful of pixels dominate the loss.
present = freq[freq > 0]
CLASS_WEIGHTS = np.clip(np.sqrt(np.median(present) / np.maximum(freq, 1e-6)), 0.5, 4.0).astype(np.float32)
print("class weights:", {n: round(float(w), 2) for n, w in zip(CLASS_NAMES, CLASS_WEIGHTS)})

## 6. Dataset and loaders

Online augmentation is a random 90-degree rotation and flip applied identically to both images and both label maps, plus a mild brightness/contrast jitter applied to **each date's image independently** (imagery only -- never the label maps, which is exactly the mistake in the dataset's own offline copies) so the model learns to ignore the global radiometric differences between two acquisitions instead of reading them as change. No temporal swap: real users pass (earlier, later), and transition priors differ by direction.

In [ ]:
MEAN = np.array((0.485, 0.456, 0.406), dtype=np.float32)
STD = np.array((0.229, 0.224, 0.225), dtype=np.float32)

def photometric_jitter(img):
    contrast, brightness = random.uniform(0.8, 1.2), random.uniform(-20.0, 20.0)
    return np.clip((img.astype(np.float32) - 128.0) * contrast + 128.0 + brightness, 0, 255).astype(np.uint8)

class SecondCCPairs(Dataset):
    def __init__(self, entries, augment):
        self.entries, self.augment = entries, augment

    def __len__(self):
        return len(self.entries)

    def _read(self, x, kind, side):
        im = Image.open(f"{ROOT}/{x['split']}/{kind}/{side}/{x['filename']}").convert("RGB")
        if im.size != (IMG_SIZE, IMG_SIZE):
            im = im.resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR if kind == "rgb" else Image.NEAREST)
        return np.asarray(im)

    def __getitem__(self, i):
        x = self.entries[i]
        a, b = self._read(x, "rgb", "A"), self._read(x, "rgb", "B")
        la, lb = decode_sem(self._read(x, "sem", "A")), decode_sem(self._read(x, "sem", "B"))
        if self.augment:
            k = random.randint(0, 3)
            a, b, la, lb = [np.rot90(t, k) for t in (a, b, la, lb)]
            if random.random() < 0.5:
                a, b, la, lb = [t[:, ::-1] for t in (a, b, la, lb)]
            a, b = photometric_jitter(a), photometric_jitter(b)
        changed = ((la != IGNORE_INDEX) | (lb != IGNORE_INDEX)).astype(np.float32)

        def to_tensor(img):
            return torch.from_numpy(((img.astype(np.float32) / 255.0 - MEAN) / STD).transpose(2, 0, 1).copy())

        return (to_tensor(a), to_tensor(b),
                torch.from_numpy(np.ascontiguousarray(la)).long(),
                torch.from_numpy(np.ascontiguousarray(lb)).long(),
                torch.from_numpy(np.ascontiguousarray(changed))[None])

BATCH = 4 if SMOKE_TEST else 16
WORKERS = 0 if SMOKE_TEST else 2
def make_loader(entries, augment, shuffle):
    return DataLoader(SecondCCPairs(entries, augment), batch_size=BATCH, shuffle=shuffle, drop_last=augment,
                      num_workers=WORKERS, pin_memory=(DEVICE == "cuda"), persistent_workers=(WORKERS > 0))

train_loader = make_loader(train_entries, augment=True, shuffle=True)
val_loader = make_loader(val_entries, augment=False, shuffle=False)
test_loader = make_loader(test_entries, augment=False, shuffle=False)
print(f"train batches={len(train_loader)}  val batches={len(val_loader)}  test batches={len(test_loader)}")

## 7. Loss and metrics

**SeK** follows the SECOND-benchmark definition: build a 7-class map per date (0 = no change, 1..6 = class), accumulate one confusion matrix over both dates, take Cohen's kappa over the changed-class cells (`kappa_n0`), and combine it with the change-class IoU as `SeK = kappa_n0 * exp(IoU_change) / e`. **Score = 0.3 * mIoU(change/no-change) + 0.7 * SeK**.

**Class-delta metrics** are this tool's own, because they're what a user is actually told: for each pair, predicted vs. true net area change per class (fraction of the scene), and for buildings whether increased / decreased / unchanged is called correctly (unchanged = under 0.5% of the scene, `NET_TOL`).

In [ ]:
NET_TOL = 0.005  # same tolerance the tool uses to call a class "unchanged" (semantic_change_tool.NET_CHANGE_TOLERANCE)
BUILDING = CLASS_NAMES.index("buildings")

def dice_loss(logits, target, eps=1.0):
    p = torch.sigmoid(logits.float())
    inter = (p * target).sum((1, 2, 3))
    return 1 - ((2 * inter + eps) / (p.sum((1, 2, 3)) + target.sum((1, 2, 3)) + eps)).mean()

class_weights_t = torch.tensor(CLASS_WEIGHTS, device=DEVICE)

def scd_loss(change, sem_a, sem_b, change_t, la, lb):
    loss = F.binary_cross_entropy_with_logits(change.float(), change_t) + dice_loss(change, change_t)
    for logits, lab in ((sem_a, la), (sem_b, lb)):
        if (lab != IGNORE_INDEX).any():  # a batch with zero labeled pixels would make the mean NaN
            loss = loss + F.cross_entropy(logits.float(), lab, weight=class_weights_t, ignore_index=IGNORE_INDEX)
    return loss

def kappa(hist):
    n = hist.sum()
    if n == 0:
        return 0.0
    po = np.trace(hist) / n
    pe = float((hist.sum(0) * hist.sum(1)).sum()) / n ** 2
    return 0.0 if pe == 1 else float((po - pe) / (1 - pe))

def scd_metrics(hist7):
    fg = hist7[1:, 1:]
    c2 = np.zeros((2, 2))
    c2[0, 0] = hist7[0, 0]
    c2[0, 1] = hist7[0].sum() - hist7[0, 0]
    c2[1, 0] = hist7[:, 0].sum() - hist7[0, 0]
    c2[1, 1] = fg.sum()
    h0 = hist7.astype(np.float64).copy()
    h0[0, 0] = 0
    kappa_n0 = kappa(h0)
    iu = np.diag(c2) / np.maximum(c2.sum(1) + c2.sum(0) - np.diag(c2), 1)
    sek = kappa_n0 * math.exp(iu[1]) / math.e
    miou = float((iu[0] + iu[1]) / 2)
    return {"IoU_change": float(iu[1]), "IoU_nochange": float(iu[0]), "mIoU": miou, "SeK": float(sek), "Score": float(0.3 * miou + 0.7 * sek)}

def direction(net):
    return np.where(net > NET_TOL, 2, np.where(net < -NET_TOL, 0, 1))  # 0 decreased, 1 unchanged, 2 increased

@torch.no_grad()
def evaluate(model, loader, threshold=0.5):
    model.eval()
    K = NUM_CLASSES
    hist7 = torch.zeros(K + 1, K + 1, dtype=torch.long, device=DEVICE)
    hist_sem = torch.zeros(K, K, dtype=torch.long, device=DEVICE)
    pred_net, gt_net = [], []
    for a, b, la, lb, chg in loader:
        a, b, la, lb = [t.to(DEVICE, non_blocking=True) for t in (a, b, la, lb)]
        change, sem_a, sem_b = model(a, b)
        pc = torch.sigmoid(change[:, 0].float()) >= threshold  # [N,H,W]
        pa, pb = sem_a.argmax(1), sem_b.argmax(1)
        for pred_cls, lab in ((pa, la), (pb, lb)):
            gt7 = torch.where(lab == IGNORE_INDEX, torch.zeros_like(lab), lab + 1)
            pr7 = torch.where(pc, pred_cls + 1, torch.zeros_like(pred_cls))
            hist7 += torch.bincount((gt7 * (K + 1) + pr7).flatten(), minlength=(K + 1) ** 2).view(K + 1, K + 1)
            m = lab != IGNORE_INDEX
            hist_sem += torch.bincount((lab[m] * K + pred_cls[m]).flatten(), minlength=K * K).view(K, K)

        hw = pc.shape[1] * pc.shape[2]
        def class_counts(cls, mask):
            return (F.one_hot(cls.clamp(max=K - 1), K) * mask.unsqueeze(-1)).sum((1, 2))
        pred_net.append(((class_counts(pb, pc) - class_counts(pa, pc)).float() / hw).cpu())
        gt_net.append(((class_counts(lb, lb != IGNORE_INDEX) - class_counts(la, la != IGNORE_INDEX)).float() / hw).cpu())

    metrics = scd_metrics(hist7.cpu().numpy())
    hs = hist_sem.cpu().numpy()
    sem_iou = np.diag(hs) / np.maximum(hs.sum(0) + hs.sum(1) - np.diag(hs), 1)
    support = hs.sum(1) > 0
    metrics["sem_mIoU"] = float(sem_iou[support].mean()) if support.any() else 0.0
    metrics["sem_IoU"] = {n: round(float(v), 4) for n, v, s in zip(CLASS_NAMES, sem_iou, support) if s}

    pn, gn = torch.cat(pred_net).numpy(), torch.cat(gt_net).numpy()
    metrics["net_MAE_pct"] = {n: round(float(np.abs(pn[:, i] - gn[:, i]).mean() * 100), 3) for i, n in enumerate(CLASS_NAMES)}
    pd_, gd_ = direction(pn[:, BUILDING]), direction(gn[:, BUILDING])
    conf = np.zeros((3, 3), dtype=np.int64)
    np.add.at(conf, (gd_, pd_), 1)  # rows = true direction, cols = predicted
    moved = gd_ != 1
    metrics["building_direction_acc"] = float((pd_ == gd_).mean())
    metrics["building_direction_acc_when_it_really_changed"] = float((pd_[moved] == gd_[moved]).mean()) if moved.any() else float("nan")
    metrics["building_direction_confusion"] = conf.tolist()  # order: decreased, unchanged, increased
    return metrics

## 8. Train

AdamW with a short linear warm-up then cosine decay, mixed precision on GPU. Validation each epoch on the val originals; the **best-Score weights are written to disk every time they improve**, so an interrupted or cancelled session still leaves a usable checkpoint (unlike the cancelled grounding v2 run, which never checkpointed before being cut).

In [ ]:
EPOCHS = 2 if SMOKE_TEST else 60
LR = 2e-4
model = SiameseSCDNet(encoder_name="resnet34", encoder_weights=None if SMOKE_TEST else "imagenet").to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
steps_total = EPOCHS * len(train_loader)
warmup = max(1, min(300, steps_total // 10))
def lr_factor(step):
    if step < warmup:
        return (step + 1) / warmup
    progress = (step - warmup) / max(1, steps_total - warmup)
    return 0.01 + 0.99 * 0.5 * (1 + math.cos(math.pi * progress))
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_factor)
use_amp = DEVICE == "cuda"
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

OUT_DIR = "/tmp/output_model" if SMOKE_TEST else "/kaggle/working/output_model"
os.makedirs(OUT_DIR, exist_ok=True)
CKPT_PATH = os.path.join(OUT_DIR, "semantic_change_unet.pt")

def to_py(obj):
    if isinstance(obj, dict):
        return {k: to_py(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [to_py(v) for v in obj]
    if isinstance(obj, (np.floating, float)):
        return float(obj)
    if isinstance(obj, (np.integer, int)):
        return int(obj)
    return obj

def save_checkpoint(state_dict, metrics, threshold):
    torch.save(
        {
            "model_state_dict": {k: v.detach().cpu() for k, v in state_dict.items()},
            "encoder_name": "resnet34",
            "img_size": IMG_SIZE,
            "class_names": list(CLASS_NAMES),
            "change_threshold": float(threshold),
            "metrics": to_py(metrics),
        },
        CKPT_PATH,
    )

best_score, best_state = -1.0, None
t_start = time.time()
step = 0
for epoch in range(EPOCHS):
    model.train()
    running, n_batches, t_epoch = 0.0, 0, time.time()
    for a, b, la, lb, chg in train_loader:
        a, b, la, lb, chg = [t.to(DEVICE, non_blocking=True) for t in (a, b, la, lb, chg)]
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=use_amp):
            change, sem_a, sem_b = model(a, b)
        loss = scd_loss(change, sem_a, sem_b, chg, la, lb)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        step += 1
        running += loss.item()
        n_batches += 1

    val = evaluate(model, val_loader)
    marker = ""
    if val["Score"] > best_score:
        best_score = val["Score"]
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        save_checkpoint(best_state, {"val_Score": val["Score"], "epoch": epoch + 1}, 0.5)
        marker = "  <- best, checkpoint written"
    print(f"epoch {epoch + 1:>2}/{EPOCHS}  loss={running / max(n_batches, 1):.4f}  val: IoU_change={val['IoU_change']:.3f} "
          f"SeK={val['SeK']:.4f} Score={val['Score']:.4f} sem_mIoU={val['sem_mIoU']:.3f}  ({time.time() - t_epoch:.0f}s){marker}", flush=True)
print(f"\ntraining took {(time.time() - t_start) / 60:.1f} min; best val Score {best_score:.4f}")

## 9. Pick the change threshold on val, then evaluate once on the untouched test split

The threshold is chosen on **val only** (the test split isn't used for any decision) and stored in the checkpoint so the tool uses the same one.

In [ ]:
model.load_state_dict(best_state)
sweep = {}
for th in (0.3, 0.4, 0.5, 0.6, 0.7):
    sweep[th] = evaluate(model, val_loader, threshold=th)["Score"]
    print(f"  threshold {th}: val Score {sweep[th]:.4f}")
BEST_TH = max(sweep, key=sweep.get)
print("chosen threshold:", BEST_TH)

val_final = evaluate(model, val_loader, threshold=BEST_TH)
test_final = evaluate(model, test_loader, threshold=BEST_TH)
print("\n=== TEST (official split, leak-filtered training) ===")
for k in ("IoU_change", "IoU_nochange", "mIoU", "SeK", "Score", "sem_mIoU"):
    print(f"  {k:14s} {test_final[k]:.4f}")
print("  per-class semantic IoU (on truly-changed pixels):", test_final["sem_IoU"])
print("  net-area-change MAE, percentage points of the scene:", test_final["net_MAE_pct"])
print(f"  buildings direction accuracy: {test_final['building_direction_acc']:.3f} overall, "
      f"{test_final['building_direction_acc_when_it_really_changed']:.3f} when buildings really did increase/decrease")
print("  buildings direction confusion (rows true, cols predicted; decreased/unchanged/increased):")
for name, row in zip(("decreased", "unchanged", "increased"), test_final["building_direction_confusion"]):
    print(f"    {name:>9s}: {row}")

## 10. Export

Same idea as the water U-Net: a plain `torch.save`d dict of tensors + builtins, so `semantic_change_tool.py` can load it with `weights_only=True`.

In [ ]:
metrics_out = {
    "val_Score": val_final["Score"], "val_SeK": val_final["SeK"],
    "test_Score": test_final["Score"], "test_SeK": test_final["SeK"], "test_IoU_change": test_final["IoU_change"],
    "test_sem_mIoU": test_final["sem_mIoU"], "test_sem_IoU": test_final["sem_IoU"],
    "test_net_MAE_pct": test_final["net_MAE_pct"],
    "test_building_direction_acc": test_final["building_direction_acc"],
    "test_building_direction_acc_when_it_really_changed": test_final["building_direction_acc_when_it_really_changed"],
    "epochs": EPOCHS, "train_pairs": len(train_entries),
}
save_checkpoint(best_state, metrics_out, BEST_TH)
print("Exported to", CKPT_PATH, f"({os.path.getsize(CKPT_PATH) / 1e6:.0f} MB) -- download it from this notebook's Output tab.")

## Next

Download `semantic_change_unet.pt` from the Output tab into `models/change_detection/checkpoints/`, then sanity-check it on a real before/after pair:

```bash
python models/change_detection/semantic_change_tool.py --image1 before.png --image2 after.png \
    --checkpoint models/change_detection/checkpoints/semantic_change_unet.pt --draw out.jpg
```

Restart the backend afterwards: the change-detection adapter picks Stage 2 automatically when that checkpoint exists (and falls back to Stage 1 pixel differencing when it doesn't).